# APEX Optimizer Tutorial: Math Problem Solving

This tutorial demonstrates how to use **APEX** (Analysis-based Prompt Engineering eXpert) to improve GPT-5 Mini's performance on AIME math problems through systematic prompt optimization.

APEX analyzes failures, recognizes success patterns, generates hypotheses, and validates improvements empirically.

## Configuration

All modifiable parameters in one place for easy adjustment:

In [1]:
# API Configuration
api_key = 'sk-12345'# Will prompt if not set
base_url = 'https://nexus-master.lmndstaging.com'

# Student Model Configuration (model being optimized)
student_model = "litellm_proxy/openai/gpt-5-mini"
student_base_url = base_url  # Optional custom API endpoint
student_temperature = 0.0  # Deterministic for math
student_reasoning_effort = 'minimal'

# Analysis Model Configuration (for failure analysis and hypotheses)
analysis_model = "litellm_proxy/openai/gpt-5"
analysis_base_url = base_url  # Optional custom API endpoint
analysis_temperature = 1.0  # Creative for hypothesis generation
analysis_reasoning_effort = 'minimal'

# APEX Optimization Settings
max_iterations = 50
num_hypotheses = 3
train_sample_size = 10
success_threshold = 1.0
convergence_patience = 10
num_threads = 50
seed = 42
verbosity = "detailed"

# MLflow Tracking (Optional)
use_mlflow = True
mlflow_tracking_uri = "http://localhost:5005"
mlflow_experiment_name = "APEX-AIME-Math"

## Setup

Import dependencies and configure language models:

In [11]:
import os
import dspy
from dspy.adapters import JSONAdapter

if api_key is None:
    api_key = os.getenv("OPENAI_API_KEY") or input("Enter your OpenAI API key: ")

# Configure student model
student_kwargs = {
    "model": student_model,
    "api_key": api_key,
    "temperature": student_temperature,
}
if student_base_url:
    student_kwargs["base_url"] = student_base_url
if student_reasoning_effort:
    student_kwargs["reasoning_effort"] = student_reasoning_effort

student_lm = dspy.LM(**student_kwargs)

# Configure analysis model
analysis_kwargs = {
    "model": analysis_model,
    "api_key": api_key,
    "temperature": analysis_temperature,
}
if analysis_base_url:
    analysis_kwargs["base_url"] = analysis_base_url
if analysis_reasoning_effort:
    analysis_kwargs["reasoning_effort"] = analysis_reasoning_effort

analysis_lm = dspy.LM(**analysis_kwargs)

analysis_adapter = JSONAdapter()
hypothesis_adapter = JSONAdapter()

dspy.configure(lm=student_lm)
dspy.settings.configure(num_threads=num_threads)

## Dataset

Load AIME problems (American Invitational Mathematics Examination):

In [12]:
from datasets import load_dataset
import random

def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            "solution": x['solution'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]
    
    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 5

    return train_set, val_set, test_set

train_set, val_set, test_set = init_dataset()

print(f"Training: {len(train_set)} | Validation: {len(val_set)} | Test: {len(test_set)}")

KeyboardInterrupt: 

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## an...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Example problem:

In [ ]:
print("Problem:", train_set[0]['problem'])
print("\nAnswer:", train_set[0]['answer'])

Problem: In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.

Answer: 242


## Program

Define a Chain of Thought program:

In [ ]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

## Metrics

Define evaluation metrics:

In [ ]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        return 0
    return int(correct_answer == llm_answer)

In [ ]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        feedback_text = (
            f"The final answer must be a valid integer. "
            f"You responded with '{prediction.answer}', which couldn't be parsed. "
            f"The correct answer is '{correct_answer}'."
        )
        
        if written_solution:
            feedback_text += f" Here's the full solution:\n{written_solution}"
        
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)
    
    if score == 1:
        feedback_text = f"Correct! The answer is '{correct_answer}'."
    else:
        feedback_text = f"Incorrect. The correct answer is '{correct_answer}'."

    if written_solution:
        feedback_text += f" Here's the full solution:\n{written_solution}"

    return dspy.Prediction(score=score, feedback=feedback_text)

## Baseline Evaluation

Evaluate the unoptimized program:

In [ ]:
eval_kwargs = dict(
    num_threads=num_threads,
    display_progress=True,
    display_table=5,
    provide_traceback=False,
)

evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    **eval_kwargs,
)

print("Evaluating baseline...")
baseline_result = evaluate(program)

print(f"\nBaseline Performance: {baseline_result.score / 100.:.1%}")

Evaluating baseline...
Average Metric: 80.00 / 150 (53.3%): 100%|██████████| 150/150 [00:01<00:00, 144.77it/s]

2025/10/17 21:13:43 INFO dspy.evaluate.evaluate: Average Metric: 80 / 150 (53.3%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,We interpret 17_b = b+7 and 97_b = 9b+7. We need b+7 to divide 9b+...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"Set up affine coordinates with A=(0,0), B=(1,0), C=(0,1). Points o...",441,✔️ [0]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,"We must count assignments of 3 labeled flavors (C, V, S) to 9 dist...",16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We need integer solutions (x,y) in [-100,100] satisfying 12x^2 - x...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Divisibility by 22 means divisible by 2 and 11. Units digit must b...,279,✔️ [1]



Baseline Performance: 53.3%


## APEX Optimization

Optimize the program with APEX:

In [ ]:
from dspy.teleprompt.apex import APEX

optimizer = APEX(
    metric=metric_with_feedback,
    analysis_lm=analysis_lm,
    hypothesis_lm=analysis_lm,
    analysis_adapter=analysis_adapter,
    hypothesis_adapter=hypothesis_adapter,
    max_iterations=max_iterations,
    num_hypotheses=num_hypotheses,
    num_eval_runs=1,
    train_sample=train_sample_size,
    success_threshold=success_threshold,
    convergence_patience=convergence_patience,
    num_threads=num_threads,
    verbosity=verbosity,
    seed=seed,
    use_mlflow=use_mlflow,
    mlflow_tracking_uri=mlflow_tracking_uri,
    mlflow_experiment_name=mlflow_experiment_name,
)

print("Starting optimization...")
optimized_program = optimizer.compile(
    student=program,
    trainset=train_set,
    valset=val_set,
)

print("\nOptimization complete!")

2025/10/17 21:13:43 INFO dspy.teleprompt.apex.apex: APEX: MLflow tracking enabled


Starting optimization...


2025/10/17 21:13:43 INFO dspy.teleprompt.apex.apex: APEX: running with num_threads=50
2025/10/17 21:13:43 INFO dspy.teleprompt.apex.apex: APEX: Configuration - max_iterations=50, num_hypotheses=3, success_threshold=1.00, convergence_patience=10
2025/10/17 21:13:43 INFO dspy.teleprompt.apex.apex: APEX: Using seed=42 for reproducibility
2025/10/17 21:13:43 INFO dspy.teleprompt.apex.apex: APEX: Evaluating initial baseline on validation set


Processed 45 / 45 examples: 100%|██████████| 45/45 [00:01<00:00, 27.14it/s]

2025/10/17 21:13:45 INFO dspy.teleprompt.apex.apex: APEX: Initial baseline score=0.5111


...

2025/10/17 21:50:48 WARNING dspy.utils.parallelizer: SIGINT received. Cancelling.
2025/10/17 21:50:48 INFO dspy.teleprompt.apex.apex: APEX: Optimization interrupted by user (Ctrl+C)


  0%|          | 0/45 [00:02<?, ?it/s]

2025/10/17 21:50:48 INFO dspy.teleprompt.apex.apex: APEX: ✓ Optimization complete | 4 iterations | Reason: interrupted
2025/10/17 21:50:48 INFO dspy.teleprompt.apex.apex: APEX: Final score: 0.5111 (+0.0000 from baseline 0.5111)
2025/10/17 21:50:48 INFO dspy.teleprompt.apex.apex: APEX: Summary - evaluated 16 candidates from 12 hypotheses
2025/10/17 21:50:48 INFO dspy.teleprompt.apex.apex: APEX: Best score trajectory across iterations: [0.5111111111111111, 0.5111111111111111, 0.5111111111111111, 0.5111111111111111]



🏃 View run omniscient-goose-264 at: http://localhost:5005/#/experiments/1/runs/6e4762a0738b46129d2b92f1716c9421
🧪 View experiment at: http://localhost:5005/#/experiments/1

╔═══════════════════════════════════════════════════════════════╗
║                    APEX Optimization Summary                     ║
╠═══════════════════════════════════════════════════════════════╣
║ Total Iterations: 4                                               ║
║ Hypotheses Tested: 12                                             ║
║ Candidates Evaluated: 16                                          ║
╠═══════════════════════════════════════════════════════════════╣
║ Initial Score: 0.5111                                             ║
║ Final Score: 0.5111                                               ║
║ Improvement: +0.0000                                              ║
╠═══════════════════════════════════════════════════════════════╣
║ Iter │ Train F/S │ Hypotheses │ Best Score │ Δ from prev ║
╟──────┼────

[Trace(trace_id=tr-41dfb9cc38259bcf196a05808d0e8e1a), Trace(trace_id=tr-5a8320effe316712f9696e7b17c71353), Trace(trace_id=tr-0ad000906b3fbee776934a30f754064f), Trace(trace_id=tr-7aa3397aa89f018ea147b91ce4ce5228), Trace(trace_id=tr-20f24fa59d9b19add7818cc3cb97983f), Trace(trace_id=tr-9e0f999dc9573423cb715106e974ee08), Trace(trace_id=tr-5dc5e42dff849201949df471a8d0bb20), Trace(trace_id=tr-c8f6e66bc2d413a4aeea3d78bfc49f1a), Trace(trace_id=tr-e0738ca6defdba99800cec08bdff7245), Trace(trace_id=tr-11166cf4d1a1263a75c86619c0224138)]

Inspect the optimized prompt:

In [ ]:
print("Optimized Prompt:")
print("=" * 50)
print(optimized_program.predict.signature.instructions)
print("=" * 50)

Optimized Prompt:
You are solving a single-step quantitative reasoning problem. Produce exactly one JSON object with two keys: "reasoning" and "answer".

Rules (must follow all):
1) Reasoning: concise, deterministic steps only; no speculation. End the reasoning with the exact sentence: "Final result = FINAL_RESULT" where FINAL_RESULT is the final integer you computed.
2) Answer: digits-only string matching FINAL_RESULT (regex ^\d+$). No spaces, signs, decimals, fractions, radicals, or braces.
3) Output format: a single flat JSON object on one line with exactly these two keys and nothing else.

Tiny verification checklist before finalizing (apply to any problem type):
- Partition/coverage: ensure all required cases or parameter classes (e.g., gcd/residue/geometry cases) are enumerated with no double-counting.
- Integer finalization: if the problem requires an integer, simplify until the result is an integer; do not output fractions/surds. If an intermediate yields a fraction m/n but the

## Final Evaluation

Evaluate the optimized program:

In [ ]:
print("Evaluating optimized program...")
optimized_result = evaluate(optimized_program)

print(f"\n{'='*50}")
print(f"Baseline:  {baseline_result.score/100.:.1%}")
print(f"Optimized: {optimized_result.score/100.:.1%}")
print(f"Improvement: {(optimized_result.score - baseline_result.score)/100.:.1%}")
print(f"{'='*50}")

Evaluating optimized program...
  0%|          | 0/150 [00:00<?, ?it/s]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{"reason...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{"reason...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpec

Average Metric: 8.00 / 12 (66.7%):   7%|▋         | 11/150 [00:14<01:01,  2.28it/s]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{"reason...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Average Metric: 44.00 / 99 (44.4%):  65%|██████▌   | 98/150 [00:33<00:04, 12.80it/s]

...

Average Metric: 56.00 / 148 (37.8%): 100%|██████████| 150/150 [01:54<00:00,  1.31it/s]

2025/10/17 02:53:31 INFO dspy.evaluate.evaluate: Average Metric: 56.0 / 150 (37.3%)


,problem,example_answer,reasoning,pred_answer,metric,answer
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70.0,Interpret numbers in base b: 17_b = 1*b + 7 = b+7. 97_b = 9*b + 7....,70,✔️ [1.000],NaN
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588.0,Place A at origin and let AB lie on x-axis and AC on y-axis for ar...,588,✔️ [1.000],NaN
2,The 9 members of a baseball team went to an ice-cream parlor after...,16.0,"We count surjective assignments of 9 labeled players to flavors C,...",292,✔️ [0.000],NaN
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117.0,"We need integer solutions (x,y) with -100≤x,y≤100 satisfying 12x^2...",117,✔️ [1.000],NaN
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279.0,A number is divisible by 22 iff divisible by 2 and 11. Divisible b...,279,✔️ [1.000],NaN



Baseline:  53.3%
Optimized: 37.3%
Improvement: -16.0%


[Trace(trace_id=tr-c972644628594d9a9ac52838a56a92d7), Trace(trace_id=tr-e163cd1b5cc0e666b540d29d70504405)]

## Optimization Insights

Examine the optimization process:

In [ ]:
if hasattr(optimized_program, 'apex_result'):
    result = optimized_program.apex_result
    
    print("Summary:")
    print(f"  Iterations: {len(result.iterations)}")
    print(f"  Candidates evaluated: {len(result.all_candidates)}")
    print(f"  Stop reason: {result.stopped_after}")
    print(f"  Best score: {result.best_candidate.overall_score:.4f}")
    
    print("\nIteration Progress:")
    for it in result.iterations:
        print(f"  Iteration {it.iteration}: {it.num_failures} failures, {len(it.hypotheses)} hypotheses, {len(it.candidates)} candidates")
    
    if result.best_candidate.hypothesis:
        h = result.best_candidate.hypothesis
        print(f"\nBest Hypothesis:")
        print(f"  Strategy: {h.strategy if hasattr(h, 'strategy') else 'N/A'}")
        print(f"  Impact Score: {h.impact_score if hasattr(h, 'impact_score') else 'N/A'}")

Summary:
  Iterations: 10
  Candidates evaluated: 21
  Stop reason: patience
  Best score: 0.5111

Iteration Progress:
  Iteration 1: 6 failures, 1 hypotheses, 2 candidates
  Iteration 2: 10 failures, 1 hypotheses, 2 candidates
  Iteration 3: 19 failures, 1 hypotheses, 2 candidates
  Iteration 4: 15 failures, 1 hypotheses, 2 candidates
  Iteration 5: 19 failures, 1 hypotheses, 2 candidates
  Iteration 6: 20 failures, 1 hypotheses, 2 candidates
  Iteration 7: 11 failures, 1 hypotheses, 2 candidates
  Iteration 8: 16 failures, 1 hypotheses, 2 candidates
  Iteration 9: 17 failures, 1 hypotheses, 2 candidates
  Iteration 10: 12 failures, 1 hypotheses, 2 candidates


## Conclusion

APEX systematically optimizes prompts through:
1. Analyzing failures to understand root causes
2. Recognizing successful patterns
3. Generating data-driven hypotheses
4. Validating improvements empirically

Try adjusting the configuration parameters to explore different optimization strategies.